### 第 07 章 Actor-Critic 方法

#### 本章概述

本章将介绍 Actor-Critic（演员-评论家）架构。它是强化学习中最重要的方法之一，结合了基于价值（Value-based）和基于策略（Policy-based）两类算法的优点。

**学习目标**：
- 理解 Actor-Critic 的基本架构和动机。
- 掌握优势函数（Advantage Function）在减小方差中的作用。
- 理解 A2C（同步）和 A3C（异步）算法的原理与实现差异。
- 深入理解并推导广义优势估计（GAE），掌握如何在偏差和方差之间取得完美平衡。

**前置知识**：
- 策略梯度定理（见第 6 章）。
- 状态价值函数 $V(s)$、动作价值函数 $Q(s, a)$ 及 TD 误差（见第 3、4 章）。\n

##### 7.1 Actor-Critic 架构

**理论部分**

在上一章的 REINFORCE 算法中，我们通过蒙特卡洛采样完整轨迹，并使用实际的折扣累积回报 $G_t$ 来更新策略。虽然引入 Baseline 降低了方差，但蒙特卡洛更新仍然面临以下问题：
1. **回合更新（Episodic）**：必须等一整个回合结束才能获得实际的 $G_t$ 并进行更新，无法实现单步（在线/Online）更新。
2. **高方差（High Variance）**：$G_t$ 包含了未来所有的随机奖励和状态转移的噪声，方差极其巨大。

为了解决这些问题，我们引入**Actor-Critic**架构：
- **Actor（演员）**：即策略网络 $\pi_\theta(a|s)$，负责根据当前状态选择动作。它通过 Critic 提供的反馈来更新自身的策略参数 $\theta$。
- **Critic（评论家）**：即价值网络（通常是状态价值 $V_\omega(s)$），负责评估 Actor 在当前状态下采取某个动作的好坏。它通过环境给予的真实奖励 $r$，利用时间差分（TD）方法更新自身的参数 $\omega$。

用 Critic 估计的价值来代替策略梯度中的蒙特卡洛回报 $G_t$：
$$ \nabla_\theta J(\theta) = \mathbb{E}_{s \sim \rho^\pi, a \sim \pi_\theta} \left[ \nabla_\theta \log \pi_\theta(a|s) Q_\omega(s, a) \right] $$

这里我们用动作价值函数 $Q_\omega(s, a)$ 代替了 $G_t$。因为 $Q$ 函数可以通过单步 TD 目标进行近似和更新：
$$ Q(s_t, a_t) \approx r_{t+1} + \gamma V_\omega(s_{t+1}) $$
这样，Actor-Critic 方法就允许我们在每走一步（或者固定几步）后立即进行参数更新，极大加快了学习效率。

**关键要点**：
- Actor 负责学习策略（如何行动）。
- Critic 负责学习价值函数（对状态或动作做出评价）。
- Critic 使用 TD 学习，引入了一定的偏差（Bias，因为依赖了自身的估计），但大幅降低了方差（Variance），并支持在线更新。\n

##### 7.2 优势函数（Advantage Function）

**理论部分**

如果直接使用 $Q_\omega(s, a)$ 作为策略梯度的权重，由于 $Q$ 值的绝对大小变化范围可能很大，梯度的方差依然不小。上一章我们知道，减去一个状态相关的基线（Baseline）可以降低方差：
$$ \nabla_\theta J(\theta) = \mathbb{E} \left[ \nabla_\theta \log \pi_\theta(a_t|s_t) (Q(s_t, a_t) - V(s_t)) \right] $$

我们将 $Q(s_t, a_t) - V(s_t)$ 称为**优势函数（Advantage Function）**，记为 $A(s_t, a_t)$：
$$ A^{\pi}(s_t, a_t) = Q^{\pi}(s_t, a_t) - V^{\pi}(s_t) $$
它的直观含义是：“在状态 $s_t$ 下采取动作 $a_t$，比该状态下所有可选动作的平均表现好多少”。如果是正数，说明动作好于平均，应该增加其概率；如果是负数，说明不如平均，应该降低其概率。

在实际算法中，为了避免同时训练 $Q$ 网络和 $V$ 网络，我们可以利用 TD 误差来近似优势函数。回顾单步 TD 目标：
$$ Q(s_t, a_t) \approx r_{t+1} + \gamma V(s_{t+1}) $$
因此，优势函数可以近似为 **TD 误差（TD Error, $\delta_t$）**：
$$ A(s_t, a_t) \approx r_{t+1} + \gamma V_\omega(s_{t+1}) - V_\omega(s_t) = \delta_t $$

这样，我们只需要训练一个状态价值网络 $V_\omega(s)$ 即可同时获得基线（$V_\omega(s_t)$）和优势（$\delta_t$）。

**Actor 和 Critic 的更新公式**：
1. **计算 TD 误差 (Advantage)**： $\delta_t = r_{t+1} + \gamma V_\omega(s_{t+1}) (1 - done) - V_\omega(s_t)$
2. **Critic 损失**（最小化 TD 误差的平方）：
   $$ L(\omega) = \frac{1}{2} \delta_t^2 $$
3. **Actor 损失**（最大化优势的期望）：
   $$ L(\theta) = - \log \pi_\theta(a_t|s_t) \delta_t $$
   *(注意：在更新 Actor 时，$\delta_t$ 视为常数系数，不参与梯度的反向传播。)*

**关键要点**：
- 优势函数 $A(s, a)$ 衡量了特定动作相对平均水平的优劣。
- TD 误差 $\delta_t$ 是优势函数 $A(s_t, a_t)$ 的一个无偏估计。
- 通过使用 TD 误差近似 Advantage，我们可以仅用一个价值网络 $V_\omega(s)$ 完成 Actor-Critic 的更新。\n

In [ ]:
import gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# 兼容处理
np.bool8 = np.bool_

# ===============================
# 1. 定义 Actor 和 Critic 网络
# ===============================
# 实际应用中，Actor 和 Critic 往往可以共享底层的特征提取层
# 这里为了清晰，我们将它们定义在同一个 Module 中，但有不同的输出头
class ActorCriticNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(ActorCriticNetwork, self).__init__()
        # 共享特征层
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        
        # Actor 头：输出动作的概率分布
        self.actor_head = nn.Linear(hidden_dim, action_dim)
        
        # Critic 头：输出状态的价值 V(s)
        self.critic_head = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        
        action_probs = F.softmax(self.actor_head(x), dim=-1)
        state_value = self.critic_head(x)
        
        return action_probs, state_value

# ===============================
# 2. 定义 Actor-Critic 智能体 (单步更新版本)
# ===============================
class ActorCriticAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.ac_net = ActorCriticNetwork(state_dim, action_dim).to(self.device)
        # Actor 和 Critic 共用一个优化器
        self.optimizer = optim.Adam(self.ac_net.parameters(), lr=lr)
        self.gamma = gamma
        
    def select_action(self, state):
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        probs, state_value = self.ac_net(state_tensor)
        
        from torch.distributions import Categorical
        m = Categorical(probs)
        action = m.sample()
        
        return action.item(), m.log_prob(action), state_value
    
    def update(self, log_prob, state_value, reward, next_state, done):
        next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0).to(self.device)
        
        # 获取下一个状态的价值 V(s_{t+1})，计算时不需要梯度
        with torch.no_grad():
            _, next_state_value = self.ac_net(next_state_tensor)
            
        # 1. 计算 TD 目标: r + gamma * V(s')
        td_target = reward + self.gamma * next_state_value * (1 - int(done))
        
        # 2. 计算 TD 误差 (即 Advantage)
        td_error = td_target - state_value
        
        # 3. 计算 Critic 损失 (MSE)
        critic_loss = F.mse_loss(state_value, td_target)
        
        # 4. 计算 Actor 损失
        # 注意 td_error 要 detach()，它在这里作为常数权重
        actor_loss = -log_prob * td_error.detach()
        
        # 5. 总损失
        loss = actor_loss + critic_loss
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

# ===============================
# 3. 训练循环演示
# ===============================
def train_ac(env_name="CartPole-v1", max_episodes=400):
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    agent = ActorCriticAgent(state_dim, action_dim)
    rewards = []
    
    for episode in range(max_episodes):
        state = env.reset()
        if isinstance(state, tuple): state = state[0]
        
        episode_reward = 0
        done = False
        truncated = False
        
        while not (done or truncated):
            # 与环境交互
            action, log_prob, state_value = agent.select_action(state)
            step_result = env.step(action)
            if len(step_result) == 5:
                next_state, reward, done, truncated, _ = step_result
            else:
                next_state, reward, done, _ = step_result; truncated = False
            
            # 单步在线更新 (Online Update)
            agent.update(log_prob, state_value, reward, next_state, done or truncated)
            
            state = next_state
            episode_reward += reward
            
        rewards.append(episode_reward)
        if (episode + 1) % 50 == 0:
            print(f"Episode {episode+1}, Reward: {np.mean(rewards[-20:]):.2f}")
            
    env.close()
    return rewards

# 运行代码 (由于单步 AC 方差依然较大，CartPole 训练可能不稳定，更先进的 A2C/PPO 表现更好)
# ac_rewards = train_ac()
# plt.plot(ac_rewards); plt.show()

##### 7.3 A2C（Advantage Actor-Critic）

**理论部分**

基础的单步 Actor-Critic 算法（如上述代码）存在两个主要缺陷：
1. 单步 TD 估计的偏差较大。
2. 连续的状态转移之间存在很强的**时序相关性**，导致网络训练不稳定。

为了解决数据相关性，DQN 使用了经验回放缓冲区（Replay Buffer）。而在 Actor-Critic 框架下，由于策略是一直在变化的（On-policy），我们不能使用旧策略产生的回放数据。

**A2C（Advantage Actor-Critic）** 提供了一种优雅的解决方案——**多环境并行（Vectorized Environments）**。
它的核心思想是：
1. 同时创建 $N$ 个相同的环境实例（Workers）。
2. 让当前的策略网络在所有 $N$ 个环境中同时交互执行 $K$ 步。
3. 收集这 $N \times K$ 个数据样本作为一个 Batch。由于这些环境的状态是互相独立的，天然打破了数据的时序相关性。
4. 使用这批数据集中计算 Advantage，并进行一次**同步（Synchronous）**的梯度下降更新。

相比于单步 AC，A2C 利用多步返回（N-step return）和多环境并行，大幅提升了训练的稳定性和收敛速度。\n

##### 7.4 A3C（Asynchronous Advantage Actor-Critic)

**理论部分**

A3C 是 DeepMind 在 2016 年提出的一种极具影响力的算法架构。全称是**异步优势演员-评论家（Asynchronous Advantage Actor-Critic）**。

与 A2C 的区别在于其 **“异步（Asynchronous)** 的参数更新机制：
- A3C 维护一个 **全局网络（Global Network）** 。
- 创建多个并行的 **工作线程（Worker Threads）** ，每个线程有自己独立的环境和 **局部的网络副本** 。
- 每个 Worker 独立地在自己的环境中交互 $K$ 步，计算出局部网络的梯度。
- 关键点：一旦某个 Worker 计算出梯度，它就立即将该梯度**异步推送到（Push）**全局网络中进行更新。
- 更新完全局网络后，Worker 立即从全局网络 **拉取（Pull）** 最新的参数覆盖自身，继续下一轮交互。

**A3C 的优势**：
1. 多个 Worker 探索状态空间的不同部分，天然打破了数据相关性。
2. 异步机制无需等待最慢的环境执行完毕，训练速度极快，能充分利用多核 CPU。

**A2C vs A3C**：
虽然 A3C 名字更响亮，但在随后的研究与工程实践中，人们发现**同步版本的 A2C 实际上往往表现更好**。原因在于：
1. A2C 的同步机制可以把 $N$ 个环境的数据打包成一个大的 Tensor，更适合现代 GPU 的大规模矩阵并行计算。
2. A3C 中，当 Worker 推送梯度时，全局网络的参数可能已经被其他 Worker 更新过多次了（策略滞后/Staleness），这会引入额外的噪声。A2C 则保证了梯度的绝对一致性。
因此，现在的流行库（如 Stable Baselines3）通常默认实现 A2C 架构。

##### 7.5 GAE（Generalized Advantage Estimation）

**理论部分**

在 7.2 节中，我们使用单步 TD 误差来估计优势函数：
$$ \hat{A}_t^{(1)} = \delta_t = r_{t+1} + \gamma V(s_{t+1}) - V(s_t) $$
这是一种**高偏差、低方差**的估计方法（严重依赖 Critic 网络的准确性）。

如果我们往前多看几步（多步估计）：
- 两步估计：$\hat{A}_t^{(2)} = \delta_t + \gamma \delta_{t+1}$
- 三步估计：$\hat{A}_t^{(3)} = \delta_t + \gamma \delta_{t+1} + \gamma^2 \delta_{t+2}$

如果一直累加到回合结束（即蒙特卡洛估计）：
$$ \hat{A}_t^{(\infty)} = \sum_{k=0}^{\infty} \gamma^k \delta_{t+k} = G_t - V(s_t) $$
这是**低偏差、高方差**的。

**广义优势估计（GAE, Generalized Advantage Estimation）** 巧妙地将上述所有不同步数的估计进行了指数加权平均，引入了一个衰减参数 $\lambda \in [0, 1]$：
$$ \hat{A}_t^{GAE(\gamma, \lambda)} = (1 - \lambda) \left( \hat{A}_t^{(1)} + \lambda \hat{A}_t^{(2)} + \lambda^2 \hat{A}_t^{(3)} + \dots \right) $$

经过代数化简，GAE 优势函数的计算公式变得非常简洁优美：
$$ \hat{A}_t^{GAE(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l} $$

**GAE 的物理意义**：
- 当 $\lambda = 0$ 时，$\hat{A}_t^{GAE} = \delta_t$。退化为单步 TD 误差（高偏差、低方差）。
- 当 $\lambda = 1$ 时，$\hat{A}_t^{GAE} = \sum_{l=0}^{\infty} \gamma^l \delta_{t+l} = G_t - V(s_t)$。退化为蒙特卡洛优势估计（低偏差、高方差）。
- 通过调节 $\lambda \in (0, 1)$（通常取 0.95），GAE 实现了**偏差与方差的完美折中（Bias-Variance Tradeoff）**，极大地提升了策略梯度算法的稳定性和收敛速度。这也是 PPO 等当今 SOTA 算法不可或缺的核心组件。

In [ ]:
# ===============================
# GAE (广义优势估计) 的代码实现演示
# ===============================
def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """
    计算一段轨迹的 GAE 优势函数和目标回报。
    
    参数:
        rewards: 这一段交互过程收集到的奖励列表/数组
        values: Critic 网络对这一段状态的价值预测 [V(s_0), V(s_1), ..., V(s_T)]
                注意，如果是完整的轨迹，还需要预测最后一个观察到的状态的 V(s_{T})
        dones: 每个步骤是否结束的标志
        gamma: 奖励折扣因子
        lam: GAE 衰减因子 lambda
    """
    advantages = np.zeros_like(rewards, dtype=np.float32)
    gae_running = 0
    
    # 逆序计算 GAE，利用递推关系：
    # A_t = delta_t + gamma * lambda * A_{t+1}
    for t in reversed(range(len(rewards))):
        # 如果下一步是终止状态，next_value 应为 0
        next_val = values[t + 1] if t + 1 < len(values) else 0.0
        
        # 计算单步 TD Error (delta_t)
        delta = rewards[t] + gamma * next_val * (1.0 - dones[t]) - values[t]
        
        # 递推计算当前的 GAE Advantage
        gae_running = delta + gamma * lam * (1.0 - dones[t]) * gae_running
        advantages[t] = gae_running
        
    # GAE 返回的目标价值通常定义为: Target = Advantage + V(s_t)
    returns = advantages + values[:len(rewards)]
    
    # 转换为 tensor
    return torch.FloatTensor(advantages), torch.FloatTensor(returns)

# 简单的测试输出
if __name__ == "__main__":
    # 模拟 3 步的轨迹数据
    mock_rewards = np.array([1.0, 1.0, -1.0])
    mock_dones = np.array([False, False, True])
    # 多预测了一步最后一个状态的价值
    mock_values = np.array([0.5, 0.8, 0.2, 0.0]) 
    
    advs, rets = compute_gae(mock_rewards, mock_values, mock_dones)
    print("计算出的 Advantages:", advs)
    print("计算出的 Target Returns:", rets)

#### 本章小结

本章我们详细讨论了 Actor-Critic 方法：
1. **Actor-Critic** 结合了策略梯度和价值函数的优势。Actor 学习策略，Critic 学习价值以指导 Actor，支持单步更新并降低了方差。
2. **优势函数（Advantage）** $A(s, a) = Q(s, a) - V(s)$ 用于衡量动作好于平均水平的程度，常常用 **TD 误差**来近似。
3. **A2C 和 A3C** 通过引入多环境并行交互，有效打破了序列时序相关性，其中 A2C 的同步机制目前被认为更加高效和稳定。
4. **GAE** 通过引入参数 $\lambda$ 指数加权多步 TD 误差，优雅地解决了长期以来的 Bias-Variance Tradeoff 问题。

**关键概念回顾**：
- Actor 与 Critic
- Advantage Function
- 单步 TD vs 蒙特卡洛回报
- A2C / A3C
- 广义优势估计 (GAE)

**下一章预告**：
基于本章的 Actor-Critic 架构和 GAE 优势估计，下一章我们将迎来深度强化学习领域绝对的里程碑和工业界标配——信赖域策略优化（TRPO）与近端策略优化（PPO）算法。它们将彻底解决策略梯度更新步长难以控制的问题！